# NeuroDiffusion: A100 Colab Training Launchpad

This notebook handles the end-to-end pipeline: MAE Pre-training -> LDM Fine-tuning -> Image Generation.
**Persistence enabled**: All results and checkpoints are automatically saved to your Google Drive.

### Pre-requisites:
1. Ensure you have `v1-5-pruned.ckpt`, `eeg_5_95_std.pth`, and `block_splits_by_image_single.pth` uploaded directly to your project folder in Google Drive.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Setup Repository
We pull the latest code without deleting your existing work.

In [ ]:
import os
%cd /content
if not os.path.exists('NeuroDiffusion'):
    !git clone https://github.com/JoaoLucasVeras/NeuroDiffusion.git
    %cd NeuroDiffusion
    !git checkout hpc-dev
else:
    %cd NeuroDiffusion
    !git reset --hard HEAD
    !git pull origin hpc-dev

## 3. Link Datasets & Persistence
We link your project's output folders to your specific Google Drive path. **Safe**: This will not overwrite your existing checkpoints on Drive.

In [ ]:
import os
import torch
import numpy as np

# --- CONFIGURATION: Update this path if you move your folder ---
DRIVE_ROOT = '/content/drive/MyDrive/MSAI SPRING 2026/AI ML Project/NeuroDiffusion_Data'

# Ensure required output folders exist on Drive
os.makedirs(f"{DRIVE_ROOT}/results", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/exps", exist_ok=True)

# Create local folder structure in Colab if not present
!mkdir -p datasets
!mkdir -p pretrains/models

# 1. Link Input Data (Files stored directly in NeuroDiffusion_Data)
print("🔗 Linking input files...")
!ln -sf "{DRIVE_ROOT}/eeg_5_95_std.pth" datasets/eeg_5_95_std.pth
!ln -sf "{DRIVE_ROOT}/block_splits_by_image_single.pth" datasets/block_splits_by_image_single.pth
!ln -sf "{DRIVE_ROOT}/v1-5-pruned.ckpt" pretrains/models/v1-5-pruned.ckpt

# 2. Link Output Media (Persistence) - ONLY link if not already linked
print("🔗 Synchronizing with Google Drive...")
for folder in ['results', 'exps']:
    target = f"{DRIVE_ROOT}/{folder}"
    if os.path.exists(folder) and not os.path.islink(folder):
        !rm -rf {folder} # Only remove local empty placeholders
    if not os.path.exists(folder):
        !ln -s "{target}" {folder}

print(f"✅ System linked to: {DRIVE_ROOT}")

# 3. Auto-Unpack for Stage 1 (Fast check if already done)
if os.path.exists('datasets/eeg_5_95_std.pth'):
    if not os.path.exists('datasets/mne_data/sub001_chunk_000.npy'):
        print("🔨 Generating fragments for Stage 1...")
        os.makedirs('datasets/mne_data', exist_ok=True)
        loaded = torch.load('datasets/eeg_5_95_std.pth', weights_only=False)
        for i, item in enumerate(loaded['dataset']):
            np.save(f'datasets/mne_data/sub001_chunk_{i:03d}.npy', item['eeg'].numpy())
        print("✅ Success: 800 fragments generated.")
    else:
        print("✅ Implementation cache found: Skipping fragment generation.")
else:
    print("❌ ERROR: Signal file not found on Drive. Please verify DRIVE_ROOT.")

## 4. Install Dependencies

In [ ]:
!pip install einops omegaconf kornia torch-fidelity timm wandb torchmetrics natsort h5py mne transformers pytorch-lightning

## 5. Stage 1: MAE Pre-training

In [ ]:
%cd /content/NeuroDiffusion/code
!python stageA1_eeg_pretrain.py --batch_size 128 --num_epoch 500

## 6. Stage 2: Diffusion Fine-tuning

In [ ]:
import glob
ckpts = sorted(glob.glob('/content/NeuroDiffusion/results/eeg_pretrain/*/checkpoints/checkpoint.pth'))
if not ckpts:
    print("❌ ERROR: Stage 1 checkpoint not found. Did you run Stage 1 successfully?")
else:
    MAE_CHECKPOINT = ckpts[-1]
    print(f"⭐ Loading latest Stage 1 Checkpoint: {MAE_CHECKPOINT}")
    %cd /content/NeuroDiffusion/code
    !python eeg_ldm.py --pretrain_mbm_path {MAE_CHECKPOINT} --batch_size 16

## 7. Stage 3: Image Generation

In [ ]:
import glob
ckpts = sorted(glob.glob('/content/NeuroDiffusion/exps/results/generation/*/checkpoint.pth'))
if not ckpts:
    print("❌ ERROR: Stage 2 checkpoint not found.")
else:
    STAGE2_CHECKPOINT = ckpts[-1]
    print(f"⭐ Generating using: {STAGE2_CHECKPOINT}")
    %cd /content/NeuroDiffusion/code
    !python gen_eval_eeg.py --dataset EEG --model_path {STAGE2_CHECKPOINT}